In [18]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


from assignment_1.modules.models import fit_bpr, bpr_scorer, fit_transitions, transition_scorer
from assignment_1.modules.util import (
    calc_pair_accuracy,
    load_csv,
    make_pair_validation,
    predict_pairs,
    Holdout
)


DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
TASK_TRAIN_FILE = "task_5_train.csv"
TASK_TEST_FILE = "task_5_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [4]:
train.head()

,user,item,timestamp,rating
0,u8441689127,i3485821335,1196491161,5
1,u8441689127,i8500230494,1199173081,4
2,u8441689127,i5178637966,1225172657,4
3,u8441689127,i2463918075,1226306675,3
4,u8441689127,i5275768745,1228900586,4


In [5]:
test.head()

,user,item1,item2,which
0,u8441689127,i5205150317,i7136647589,NaN
1,u4846441168,i5621122193,i6971891859,NaN
2,u3746550648,i2399124454,i6926978284,NaN
3,u1248621856,i4635029767,i1417365919,NaN
4,u8157052397,i9077033308,i2794150437,NaN


In [6]:
print(f"train {train.shape}, test {test.shape}")
print(f"test users unique: {test.user.is_unique} | train users: {train.user.nunique():,}")

pop = train["item"].value_counts()
p1, p2 = test["item1"].map(pop).fillna(0), test["item2"].map(pop).fillna(0)
print(f"unseen: users {(~test.user.isin(train.user)).mean():.1%}, "
      f"item1 {(p1 == 0).mean():.1%}, item2 {(p2 == 0).mean():.1%}")
print(f"median pop: item1 {p1.median():.0f}, item2 {p2.median():.0f} | "
      f"refs: uniform {pop.median():.0f}, ∝ interactions {train['item'].map(pop).median():.0f}")
print(f"P(item1 more popular) = {(p1 > p2).mean():.3f}")

# is each user's history in time order in the file?
in_order = train.groupby("user", sort=False)["timestamp"].apply(lambda s: s.is_monotonic_increasing)
print(f"users whose rows are time-ordered: {in_order.mean():.1%}")

train (2554047, 4), test (86786, 4)
test users unique: True | train users: 86,786
unseen: users 0.0%, item1 2.0%, item2 2.0%
median pop: item1 101, item2 100 | refs: uniform 2, ∝ interactions 91
P(item1 more popular) = 0.499
users whose rows are time-ordered: 100.0%


In [7]:
tr, pairs = make_pair_validation(train, holdout=Holdout.LAST, seed=0)
print(len(pairs))     # ≈ 86k: one pair per user (minus a few dropped by the seen filter)

83508


In [8]:
u1, u2 = p1 == 0, p2 == 0
print(f"exactly one unseen: {(u1 ^ u2).mean():.2%} | both unseen: {(u1 & u2).mean():.2%}")

exactly one unseen: 3.96% | both unseen: 0.00%


In [9]:
base_fit = fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)
base = calc_pair_accuracy(pairs, bpr_scorer(base_fit))
print(f"baseline BPR val acc = {base:.4f}  (leaderboard public 0.8189)  -> 2% target ≈ {base * 1.02:.4f}")

baseline BPR val acc = 0.8425  (leaderboard public 0.8189)  -> 2% target ≈ 0.8594


In [10]:
def run_grid_t5(grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
        params = dict(zip(grid, values))
        acc = calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(tr, **params)))
        rows.append({**params, "val_acc": acc, "vs_base_%": (acc / base - 1) * 100})
    return pd.DataFrame(rows).sort_values("val_acc", ascending=False)



In [11]:
grid = {
    "factors":        [5, 20, 50],
    "regularization": [0.001, 0.01, 0.1],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

results1 = run_grid_t5(grid)
results1

  0%|          | 0/9 [00:00<?, ?it/s]

,factors,regularization,learning_rate,iterations,val_acc,vs_base_%
7,50,0.010,0.01,100,0.868599,3.095641
4,20,0.010,0.01,100,0.865163,2.687721
3,20,0.001,0.01,100,0.857499,1.778075
6,50,0.001,0.01,100,0.856936,1.711273
1,5,0.010,0.01,100,0.842518,0.000000
0,5,0.001,0.01,100,0.837261,-0.623961
2,5,0.100,0.01,100,0.810054,-3.853206
5,20,0.100,0.01,100,0.806366,-4.290973
8,50,0.100,0.01,100,0.803791,-4.596558


In [13]:
grid = {
    "factors":        [50, 100],
    "regularization": [0.005, 0.01, 0.02],
    "learning_rate":  [0.01],
    "iterations":     [100],
}
results2 = run_grid_t5(grid)
results2

  0%|          | 0/6 [00:00<?, ?it/s]

,factors,regularization,learning_rate,iterations,val_acc,vs_base_%
0,50,0.005,0.01,100,0.870396,3.308839
3,100,0.005,0.01,100,0.869569,3.210768
1,50,0.010,0.01,100,0.868599,3.095641
4,100,0.010,0.01,100,0.867689,2.987620
2,50,0.020,0.01,100,0.860037,2.079395
5,100,0.020,0.01,100,0.858517,1.898887


In [15]:
tf = fit_transitions(tr)
tf.counts.sort_values(ascending=False).head()   # most common transitions (likely series: book n -> n+1)
len(tf.counts), len(tf.last_item)               # distinct transitions; users

(1941151, 86786)

In [16]:
tscore = transition_scorer(fit_transitions(tr))
t1 = tscore(pairs["user"].to_numpy(), pairs["item1"].to_numpy())
(t1 > 0).mean()     # fraction of rows where item1 has any transition from the user's last book

np.float64(0.1779110983378838)

In [20]:
TASK5_BPR_PARAMS = {"factors": 50, "regularization": 0.005,
                    "learning_rate": 0.01, "iterations": 100}
users = pairs["user"].to_numpy()
which = pairs["which"].to_numpy()
t1 = tscore(users, pairs["item1"].to_numpy())
t2 = tscore(users, pairs["item2"].to_numpy())
has_signal = t1 != t2                                         # transitions can separate the two

t_pred = np.where(t1 >= t2, pairs["item1"], pairs["item2"])
b_pred = predict_pairs(pairs, bpr_scorer(fit_bpr(tr, **TASK5_BPR_PARAMS)))

print(f"coverage:                {has_signal.mean():.1%}")
print(f"transitions on covered:  {(t_pred == which)[has_signal].mean():.4f}")
print(f"BPR on covered:          {(b_pred == which)[has_signal].mean():.4f}")
combined = np.where(has_signal, t_pred, b_pred)
print(f"BPR alone:               {(b_pred == which).mean():.4f}")
print(f"combined:                {(combined == which).mean():.4f}")

coverage:                31.2%
transitions on covered:  0.8796
BPR on covered:          0.9009
BPR alone:               0.8704
combined:                0.8637


In [21]:
for k in [1, 2, 3, 5, 10]:
    strong = np.abs(t1 - t2) >= k
    combined = np.where(strong, t_pred, b_pred)
    print(f"k={k:<3} coverage {strong.mean():6.1%}  "
          f"trans {(t_pred == which)[strong].mean():.4f}  BPR {(b_pred == which)[strong].mean():.4f}  "
          f"combined {(combined == which).mean():.4f}")

k=1   coverage  31.2%  trans 0.8796  BPR 0.9009  combined 0.8637
k=2   coverage  22.0%  trans 0.9159  BPR 0.9140  combined 0.8708
k=3   coverage  18.2%  trans 0.9305  BPR 0.9199  combined 0.8723
k=5   coverage  14.3%  trans 0.9475  BPR 0.9289  combined 0.8731
k=10  coverage  10.3%  trans 0.9673  BPR 0.9445  combined 0.8727
